In [1]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

# Using _trajectory-container-tools_ with Panda `DataFrame`

## Import trajectory-container-tools namespace

In [2]:
import trajectory_container_tools as tct

ImportError: ROS functionality requires additional dependencies. Install TCT with: pip install trajectory-container-tools[ros] or pip install "git+https://github.com/norlab-ulaval/trajectory-container-tools.git#egg=trajectory-container-tools[ros]" and make sure a ros2 distribution is in python path i.e., source ros.

## Import dataframe

In [ ]:
dataset_path = "data/repository_data/tests_data/dataframe_test_data/marmotte/ga_hard_snow_25_01_a/slip_dataset_all.pkl"
dataset_snow, dataset_real_path = tct.extractor.unpack_dataframe_and_show_topic(dataset_path)

### Requirement:
The dataframe must contain one trajectory or a batch of trajectories (one per row) with some features (column) containing a timestep index in there name e.g., `feature_1`, `feature_2` ...

In [ ]:
dataset_snow.head()

## 1. Base case

Extract multiple features from a dataset (formated in a dataframe) based on a configuration dictionary.

The `features_config` specify the feature name to lookout in the `dataset_frame` header and agregate them in a `Multifeature` dataclass. Feature dimensions such as 'x', 'y' 'z' are specified either by using existing `AbstractFeatureDataclass` subclass such as: `StatePose2D`, `CmdStandard`, `CmdSkidSteer`, `Velocity`, `VelocitySkidSteer` or by using tuple of strings such as `('<new feature dataclass type name>', '<dimension names 1>', '<dimension names 2>', ...)`.

```python
feature_config = {
    'icp_interpolated': StatePose2D,
    'idd_vel':          StatePose2D,
    'icp':              ('StatePose3D', 'x', 'y', 'z', 'roll', 'pitch', 'yaw')
}
```

Note that each `AbstractFeatureDataclass` subclass validate that each dimension have uniform shape and have monotonic increassing timestep index without skip.


In [ ]:
from trajectory_container_tools.dataclasses import StatePose2D

mf1 = tct.extractor.from_dataframe(dataset_snow,
                         dataset_info="Robot: marmotte, Details: ga_hard_snow_25_01_a",
                         features_config={
                             'body_vel_disturption': StatePose2D,
                             'icp_interpolated': StatePose2D,
                             'icp_vel': StatePose2D,
                             'idd_vel': StatePose2D,
                             'icp':     ('StatePose3D', 'x', 'y', 'z', 'roll', 'pitch', 'yaw')
                         },
                         header_mix_label_and_timesteps=True)

print(mf1)


## 2. Case requiring post-processing

1. Just create a new dataclass inheriting from a `AbstractFeatureDataclass` subclass e.g. `StatePose2D`
2. Overide `post_init_feature_callback` with the desired post-processing process

In [ ]:
steady_state_mask = dataset_snow['steady_state_mask'].to_numpy() == True

@dataclass
class StatePose2DSteadyState(StatePose2D):

    def post_init_feature_callback(self, feature_name):
        # Example for creating an explicit timestep t=0 property named "<feature_name>_init"
        feature = self.get_dynamic_field(feature_name)
        if isinstance(feature, np.ndarray):
            if self.batch:
                # Case batch data
                feature_ini = feature[:, 0, ...]
            else:
                # Case time-serie data
                feature_ini = feature[0, ...]
            self.set_dynamic_field(f"{feature_name}_init", feature_ini)
        return None

In [ ]:
mfs = tct.extractor.from_dataframe(
        dataset_snow,
        dataset_info="Robot: marmotte, Details: ga_hard_snow_25_01_a, STEADY STATE",
        features_config={
            'body_vel_disturption': StatePose2DSteadyState,
            'icp_interpolated': StatePose2DSteadyState,
            'icp_vel': StatePose2DSteadyState,
            'idd_vel': StatePose2DSteadyState,
        },
        header_mix_label_and_timesteps=True)

## 3. Accessing Data

### You can print the summary of the data container
Dataframe data can be accessed through the structured dataclass attributes. Each message type provides access to its specific fields.

In [ ]:
print(mfs)

## You can access each features and their dimension by property call

In [ ]:
mfs.idd_vel.x.shape == mfs.icp_vel.x.shape == mfs.body_vel_disturption.x.shape

## You can print the summary of any data container features

In [ ]:
print(mfs.body_vel_disturption)

## Simplify your code and minimize risk of piping the wrong data

In [ ]:
def plot_dataset_trajectory_steady_state_x_array(trajectory_id: int, title: str, y_label:str, ylim: tuple=(-2.5, 2.5)):
    fig = plt.figure(num=None, figsize=(7,3), dpi=None, facecolor=None, edgecolor=None)
    plt.title(f'{title}   (TRJ ID {trajectory_id})')
    plt.plot(mfs.idd_vel.x[trajectory_id, :], label="idd_body_vel_x", color="green")
    plt.plot(mfs.icp_vel.x[trajectory_id, :], label="icp_body_vel_x", color="blue", linestyle="dotted")
    plt.plot(mfs.body_vel_disturption.x[trajectory_id, :], label="body_vel_disturption_x", color="red", linestyle="dashed", linewidth="2.")
    plt.legend()
    plt.ylabel(y_label)
    plt.xlabel('Timestep')
    plt.ylim(*ylim)
    return None

_ids = range(23,25)
for _trj_id in _ids:
    plot_dataset_trajectory_steady_state_x_array(trajectory_id=_trj_id, title=mfs.dataset_info, y_label="x", ylim=(-2.5, 2.5))

In [ ]:
def plot_dataset_trajectory_steady_state_yaw(trajectory_id: int, title: str, y_label:str, ylim: tuple=(-2.5, 2.5)):
    fig = plt.figure(num=None, figsize=(7,3), dpi=None, facecolor=None, edgecolor=None)
    plt.title(f'{title}   (TRJ ID {trajectory_id})')
    plt.plot(mfs.idd_vel.yaw[trajectory_id, :], label="steady_state_idd_body_vel_yaw", color="green")
    plt.plot(mfs.icp_vel.yaw[trajectory_id, :], label="steady_state_icp_body_vel_yaw", color="blue", linestyle="dotted")
    plt.plot(mfs.body_vel_disturption.yaw[trajectory_id, :], label="steady_state_body_vel_disturption_yaw", color="red", linestyle="dashed", linewidth="2.")
    plt.legend()
    plt.ylabel(y_label)
    plt.xlabel('Timestep')
    plt.ylim(*ylim)
    return None

# _ids = [29,30,31,32]
_ids = range(23,25)
for _trj_id in _ids:
    plot_dataset_trajectory_steady_state_yaw(trajectory_id=_trj_id, title=mfs.dataset_info, y_label="yaw", ylim=(-3.1416, 3.1416))